<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c42-header.png" alt="Nextia Learning · Vision AI: See, Count and Inspect" width="100%">

# Solution: Measure from masks

**[Measure from masks](https://learning.nextia-ai.com/courses/vision/m05/measure-from-masks/)** · Vision AI: See, Count and Inspect · Module 5, lesson 3 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** measure an area from a mask: count its pixels, turn pixels into square millimetres with an object of known size, and see how much the answer depends on that size; then compare two masks with IoU and Dice, and see why a one-pixel mistake ruins the score of a tiny mask.

| | |
|---|---|
| **Time** | About 35 minutes; the cells run in a few seconds. |
| **Needs** | An internet connection for the setup cells. No account, no key and no model download. |
| **You should know** | Masks, from [Semantic and instance segmentation](https://learning.nextia-ai.com/courses/vision/m05/semantic-and-instance-segmentation/), and IoU, from [Boxes and IoU](https://learning.nextia-ai.com/courses/vision/m04/boxes-and-iou/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, pandas 3.1.1, Pillow 13.0.0, scikit-learn 1.9.1 and matplotlib 3.11.2 (PyTorch 2.15.0 and timm 1.0.31 where the lesson needs them), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/vision/m05/measure-from-masks/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C42/M05-L03-measure-from-masks/measure-from-masks-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, pandas 3.1.1, pillow 13.0.0, scikit-learn 1.9.1, scikit-image 0.26.0, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 pandas==3.1.1 pillow==13.0.0 scikit-learn==1.9.1 scikit-image==0.26.0 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
import numpy; print("numpy", numpy.__version__)
import pandas; print("pandas", pandas.__version__)
import PIL; print("pillow", PIL.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)
import skimage; print("scikit-image", skimage.__version__)

## Setup: get the Larkfield images

Run the next cell. It downloads the image sets that this lesson needs (about 15 MB, as zip files) into a folder `larkfield-images`, with the course's helper `compare.py`, checks each file's checksum and unpacks it. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-images` (in Colab: the **Files** panel on the left), then run the cell again.

Where the photos come from: the catalogue photos are cut from Open Images photos (CC BY 2.0: the author of each photo is in `data/catalogue/manifest.csv`), the returned candles are photos of the VisA inspection set (CC BY 4.0), and the pallets are drawn by our code. Read `data/dataset-card.md` for the details.

In [ ]:
# Setup: download the image sets into larkfield-images/, check them and unpack them.
import hashlib, os, shutil, subprocess, urllib.request, zipfile
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C42/"
FILES = {
    "archives/returns.zip": "dcf02f521f68228d34717b303628b2db8b19d18433126da2ce64047f1449210f",
    "compare.py": "e9794acacbce00c240906cc8b3e2256b8505ff4a8074b7507a8b159cd39b4826",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-images").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-images and run again.")
    if name.endswith(".zip"):
        with zipfile.ZipFile(path) as z:
            z.extractall(PROJECT / "data")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
n = sum(1 for p in Path("data").rglob("*") if p.is_file())
print(f"Ready: {n} files in larkfield-images/data.")

## Setup: helpers

The next cell loads NumPy as `np`, pandas as `pd`, Pillow's `Image`, matplotlib, and the course helper as `cp` (the comparison table). It defines `show(paths)`, which draws photos in a row, and the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: images, the comparison table, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import compare as cp
warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
CLASSES = ["bowl", "candle", "chair", "cushion", "lamp", "mug", "plant_pot", "vase"]

def show(paths, titles=None, size=2.2):
    """Draw photos in a row (at most 8)."""
    paths = list(paths)[:8]
    fig, axes = plt.subplots(1, len(paths), figsize=(size * len(paths), size))
    for i, (ax, p) in enumerate(zip(np.atleast_1d(axes), paths)):
        ax.imshow(Image.open(p)); ax.axis("off")
        if titles:
            ax.set_title(str(titles[i]), fontsize=8)
    plt.show()

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. An area in pixels

Every damaged returns photo has a **damage mask**: a black-and-white image, the size of the photo, white where the damage is. The masks come with the VisA photos (CC BY 4.0); people drew them.

The simplest measurement: count the white pixels.

In [ ]:
returns = pd.read_csv("data/returns/manifest.csv")
row = returns.set_index("item_id").loc["R-d17d264e"]
mask = np.asarray(Image.open("data/" + row["mask_file"]).convert("L")) > 0
print(row["damage"], "|", mask.shape, "|", mask.sum(), "pixels |", f"{mask.mean():.4f} of the photo")
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
axes[0].imshow(Image.open("data/" + row["file"])); axes[0].set_title("photo")
axes[1].imshow(mask, cmap="gray"); axes[1].set_title("damage mask")
for ax in axes: ax.axis("off")
plt.show()

> **Check.** You should see 1262 pixels, 0.0136 of the photo. But "1,262 pixels" means nothing to Kwame. A pixel is not a size: a photo taken from closer has bigger pixels.

## 2. A scale from an object of known size

Every photo shows four tea lights. A standard tea light is about **38 mm** wide: an **assumption**. If we find how many pixels wide a tea light is, we know how many millimetres one pixel is.

The candles are the four largest bright regions. A region's area is pi x r x r, so its diameter is 2 x the square root of (area / pi).

In [ ]:
from scipy import ndimage
TEA_LIGHT_MM = 38.0  # an assumption: a standard tea light is about 38 mm wide

def candle_diameter_px(path):
    """The median diameter of the four largest bright regions (the candles), in pixels."""
    a = np.asarray(Image.open(path).convert("L"), float)
    bright = a > 150
    regions, n = ndimage.label(ndimage.binary_opening(bright, iterations=2))
    sizes = sorted(ndimage.sum(bright, regions, range(1, n + 1)), reverse=True)[:4]
    return float(np.median([2 * np.sqrt(s / np.pi) for s in sizes]))   # area = pi * r * r

d_px = candle_diameter_px("data/" + row["file"])
mm_per_px = TEA_LIGHT_MM / d_px
print(f"a tea light is {d_px:.1f} pixels wide, so 1 pixel is {mm_per_px:.3f} mm,")
print(f"and 1 square pixel is {mm_per_px ** 2:.4f} mm2.")
print(f"damage: {mask.sum()} pixels x {mm_per_px ** 2:.4f} = {mask.sum() * mm_per_px ** 2:.1f} mm2")

> **Check.** You should see 102.1 pixels and 175.0 mm2: the largest damage of the test photos. Note the square: if the tea light were 40 mm wide, every length grows by 40/38, and every area by (40/38) x (40/38) = 1.11.

## 3. Every damaged test photo

In [ ]:
rows = []
for _, r in returns[(returns["split"] == "test") & (returns["status"] == "damaged")].iterrows():
    m = np.asarray(Image.open("data/" + r["mask_file"]).convert("L")) > 0
    scale = TEA_LIGHT_MM / candle_diameter_px("data/" + r["file"])
    rows.append({"item_id": r["item_id"], "pixels": int(m.sum()), "mm2": round(m.sum() * scale ** 2, 1), "damage": r["damage"]})
areas = pd.DataFrame(rows).sort_values("mm2")
print(areas.to_string(index=False))
print("median", areas["mm2"].median(), "mm2")

> **Check.** 18 damaged photos, median area 29.9 mm2. The project's `vision masks` command prints the same table. Four colour spots are under 2 mm2: 4 to 10 pixels.

## 4. Compare two masks: IoU and Dice

When a model draws a mask, we compare it with the true mask. **Mask IoU** is the pixels in both, divided by the pixels in either. **Dice** is twice the pixels in both, divided by the two sizes added. Both are 1 for a perfect mask and 0 for no overlap.

In [ ]:
def mask_iou(a, b):
    """Pixels in both masks, divided by pixels in either mask."""
    return (a & b).sum() / (a | b).sum()

def dice(a, b):
    """Twice the pixels in both masks, divided by the pixels of the two masks added."""
    return 2 * (a & b).sum() / (a.sum() + b.sum())

truth = np.zeros((6, 6), bool); truth[[1, 1, 2, 2, 2, 3, 3, 4], [1, 2, 1, 2, 3, 2, 3, 3]] = True
pred = np.zeros((6, 6), bool); pred[[1, 2, 2, 2, 3, 3], [2, 1, 2, 3, 3, 4]] = True
print(truth.sum(), pred.sum(), (truth & pred).sum(), "| IoU", round(mask_iou(truth, pred), 3), "| Dice", round(dice(truth, pred), 3))

> **Check.** 8 true pixels, 6 predicted, 5 in both: IoU 5/9 = 0.556, Dice 10/14 = 0.714. Dice is always at least as high as IoU for the same masks.

### One pixel too many, all round

A model's mask is often a little too big or too small at the edge. Here, we grow each true mask by one pixel all round, and score the grown mask against the truth.

> **Predict.** Will a one-pixel mistake cost more on the largest damage or on the smallest?

In [ ]:
scores = []
for _, r in areas.iterrows():
    m = np.asarray(Image.open("data/" + returns.set_index("item_id").loc[r["item_id"], "mask_file"]).convert("L")) > 0
    grown = ndimage.binary_dilation(m)   # one pixel more on every side
    scores.append({"item_id": r["item_id"], "pixels": int(m.sum()), "iou": round(mask_iou(grown, m), 3), "dice": round(dice(grown, m), 3)})
scores = pd.DataFrame(scores)
print(scores.to_string(index=False))
plt.scatter(scores["pixels"], scores["iou"]); plt.xscale("log")
plt.xlabel("pixels in the true mask"); plt.ylabel("IoU of the mask one pixel too big"); plt.show()

> **Check.** The largest damage keeps an IoU of 0.91; the smallest (4 pixels) falls to 0.33. The same small mistake, a very different score: on tiny masks, report the score with the size, or use a distance in pixels.

## 5. Your turn: Dice of the grown masks

> **Your turn.** Put the **median** Dice of the 18 grown masks in `median_dice` (a number from 0 to 1).

In [ ]:
median_dice = float(scores["dice"].median())
print(round(median_dice, 3))

In [ ]:
expect('median_dice', median_dice, 0.857, tolerance=0.002)

## Next

The [lesson page](https://learning.nextia-ai.com/courses/vision/m05/measure-from-masks/) has the pictures, the project command and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Two ways to find damage](https://learning.nextia-ai.com/courses/vision/m06/two-ways-to-find-damage/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/vision/m05/measure-from-masks/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The photos are open photos (each with its author and licence in the manifests) or images made by our code; no photo shows a real person.